# Soil classification - XGBoost

Trains on the combined ground-truth dataset, where urban and bare-soil ground truth
have been merged into a **single label column, `soilClassifiedId`**.

**Features:** the 22 spectral bands + indices. `Longitude`, `Latitude` and `ClassID`
are **excluded** - coordinates let the model memorise locations instead of learning
spectral signatures, and `ClassID` is label-derived leakage.

**Binary or multi-class is detected automatically** from the number of distinct values
in `soilClassifiedId`; the objective, eval metrics, class weighting, decision rule and
scoring all follow from that. Nothing needs editing if the label turns out to have more
than two classes.

**Checkpointing:** a custom `TrainingCallback` writes `best_model.json` every time the
validation metric improves, so the file on disk always holds the best weights seen - not
the last ones. After training the booster is also sliced to `best_iteration` and the
reloaded model is verified to reproduce identical predictions.

Runs on Kaggle CPU or GPU, no internet required.

## 1. Config

In [1]:
# ------------------------------------------------------------------ data ----
# Tried in order. If none exist, the loader scans /kaggle/input for any csv/parquet
# whose header contains TARGET - so the dataset is found whatever the slug is called.
DATA_CANDIDATES = [
    "/kaggle/input/datasets/mezbaussalaheen/bare-soil-ground-truth-dataset/soil-dataset.parquet",
]

TARGET    = "soilClassifiedId"                    # the single merged label column
DROP_COLS = ["Longitude", "Latitude", "ClassID"]  # never enter the feature matrix

# Cosmetic only - pretty names for reports and plots, keyed by the RAW values in
# TARGET. Anything not listed falls back to "class <value>".
#   e.g. CLASS_NAMES = {0: "urban", 1: "bare soil"}
CLASS_NAMES = {}

# ----------------------------------------------------------------- split ----
# "stratified"    - random split, stratified on the label (conventional)
# "spatial_block" - whole coordinate grid-cells go to one split only. Gives an
#                   HONEST generalisation estimate, because neighbouring pixels
#                   are near-duplicates of each other.
SPLIT_STRATEGY = "stratified"
BLOCK_SIZE_DEG = 0.05        # grid cell size for spatial_block
TEST_SIZE      = 0.15
VAL_SIZE       = 0.15        # taken from what remains after the test split
SEED           = 42

# ----------------------------------------------------------------- model ----
USE_GPU            = True    # falls back to CPU automatically when unavailable
NUM_BOOST_ROUND    = 3000
EARLY_STOP_ROUNDS  = 100
BALANCE_CLASSES    = True    # scale_pos_weight (binary) / sample weights (multi-class)

# metrics are picked to match the task once the class count is known
BINARY_EARLY_STOP_METRIC = "aucpr"           # aucpr | auc | logloss | error
BINARY_EXTRA_METRICS     = ["logloss", "auc"]
MULTI_EARLY_STOP_METRIC  = "mlogloss"        # mlogloss | merror | auc
MULTI_EXTRA_METRICS      = ["merror"]

# objective / num_class / eval_metric are set in section 5 from the detected task
PARAMS = {
    "max_depth":        6,
    "eta":              0.05,          # learning rate
    "subsample":        0.8,
    "colsample_bytree": 0.8,
    "min_child_weight": 1,
    "gamma":            0.0,
    "lambda":           1.0,           # L2
    "alpha":            0.0,           # L1
    "seed":             SEED,
}

# ------------------------------------------------------------- checkpoint ----
CKPT_DIR    = "/kaggle/working/checkpoints"
BEST_MODEL  = "best_model.json"        # written mid-training on every improvement
FINAL_MODEL = "final_model.json"       # sliced booster written at the end

# ------------------------------------------------------------- inference ----
TUNE_THRESHOLD = True    # binary only - multi-class always uses argmax

## 2. Imports and data loading

In [2]:
import json, os, sys, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import xgboost as xgb

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score, confusion_matrix,
                             classification_report, roc_curve, precision_recall_curve,
                             matthews_corrcoef)

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

XGB_VERSION = tuple(int(x) for x in xgb.__version__.split(".")[:2])
print(f"xgboost {xgb.__version__}   pandas {pd.__version__}   numpy {np.__version__}")

Path(CKPT_DIR).mkdir(parents=True, exist_ok=True)
np.random.seed(SEED)

xgboost 3.2.0   pandas 2.3.3   numpy 2.0.2


In [3]:
def _read_any(p):
    return pd.read_parquet(p) if p.suffix.lower() in (".parquet", ".pq") else pd.read_csv(p)


def _has_target(p):
    """Peek at the header only - cheap even for large files."""
    try:
        if p.suffix.lower() in (".parquet", ".pq"):
            import pyarrow.parquet as pq
            return TARGET in pq.read_schema(p).names
        return TARGET in pd.read_csv(p, nrows=0).columns
    except Exception:
        return False


def load_dataset():
    for c in DATA_CANDIDATES:
        p = Path(c)
        if not p.exists():
            continue
        if not _has_target(p):
            # a stale file from an earlier version of the dataset - keep looking
            print(f"skipping {p} (no '{TARGET}' column)")
            continue
        print(f"loading {p}")
        return _read_any(p)

    root = Path("/kaggle/input")
    if not root.exists():
        raise FileNotFoundError(
            "no /kaggle/input and none of DATA_CANDIDATES exist - attach the dataset "
            "or point DATA_CANDIDATES at the file")

    files = sorted(p for p in root.rglob("*")
                   if p.is_file() and p.suffix.lower() in (".csv", ".parquet", ".pq"))
    matches = [p for p in files if _has_target(p)]
    if not matches:
        found = "\n  ".join(str(p) for p in files[:25]) or "(nothing)"
        raise FileNotFoundError(
            f"no attached file has a '{TARGET}' column. Files seen:\n  {found}")

    # several shards in one folder -> concatenate them all
    folder = matches[0].parent
    shards = [p for p in matches if p.parent == folder]
    print(f"found {len(matches)} file(s) with '{TARGET}'; loading {len(shards)} from {folder}")
    for p in shards:
        print(f"  {p.name}")
    return pd.concat([_read_any(p) for p in shards], ignore_index=True)


df = load_dataset()

print(f"\nshape: {df.shape}")
print(f"columns: {list(df.columns)}")
assert TARGET in df.columns, f"'{TARGET}' not in the dataframe"
print(f"\n{TARGET} value counts:")
print(df[TARGET].value_counts(dropna=False).sort_index())
display(df.head())

loading /kaggle/input/datasets/mezbaussalaheen/bare-soil-ground-truth-dataset/soil-dataset.parquet

shape: (3633185, 26)
columns: ['Longitude', 'Latitude', 'B01', 'B02', 'B03', 'B04', 'B05', 'B06', 'B07', 'B08', 'B8A', 'B09', 'B11', 'B12', 'EVI', 'NDVI', 'SAVI', 'NDBI', 'NDMI', 'MNDWI', 'NDWI', 'BSI', 'BI', 'AWEI', 'ClassID', 'soilClassifiedId']

soilClassifiedId value counts:
soilClassifiedId
0    1439443
1    2193742
Name: count, dtype: int64


,Longitude,Latitude,B01,B02,B03,B04,B05,B06,B07,B08,B8A,B09,B11,B12,EVI,NDVI,SAVI,NDBI,NDMI,MNDWI,NDWI,BSI,BI,AWEI,ClassID,soilClassifiedId
0,277555.0,2639715.0,0.0291,0.0872,0.1296,0.1492,0.1645,0.2295,0.2679,0.2754,0.2775,0.2567,0.2736,0.2130,0.208031,0.297221,0.204737,-0.003279,0.003279,-0.357143,-0.360000,0.076649,0.139744,-1.230600,5,1
1,277565.0,2639715.0,0.0301,0.1126,0.1586,0.1801,0.1759,0.2312,0.2464,0.2754,0.2581,0.2394,0.2961,0.2774,0.157625,0.209221,0.149608,0.036220,-0.036220,-0.302397,-0.269124,0.102060,0.169691,-1.381700,5,1
2,277575.0,2639715.0,0.0301,0.0936,0.1304,0.1482,0.1759,0.2312,0.2464,0.2593,0.2581,0.2394,0.2961,0.2774,0.192015,0.272638,0.183636,0.066259,-0.066259,-0.388511,-0.330767,0.114651,0.139584,-1.490475,5,1
3,277585.0,2639715.0,0.0294,0.0533,0.0886,0.1022,0.1388,0.2025,0.2365,0.2277,0.2574,0.2394,0.2305,0.1720,0.217708,0.380418,0.226835,0.006111,-0.006111,-0.444688,-0.439772,0.084243,0.095642,-1.097525,5,1
4,277595.0,2639715.0,0.0294,0.0364,0.0712,0.0786,0.1388,0.2025,0.2365,0.2352,0.2574,0.2394,0.2305,0.1720,0.273051,0.499044,0.288646,-0.010092,0.010092,-0.528008,-0.535248,0.064577,0.074991,-1.169000,5,1


## 3. Feature selection and label encoding

`Longitude`, `Latitude` and `ClassID` are dropped:

* **Longitude / Latitude** - a tree model splits on raw coordinates and effectively memorises
  which map tiles are which class. Accuracy looks great and the model transfers nowhere.
* **ClassID** - derived from the same annotation that produced `soilClassifiedId`, so it leaks
  the answer.

The raw label values are mapped to contiguous `0..K-1` indices (XGBoost requires that for
multi-class), and the mapping is saved with the model so predictions can be decoded back.

In [4]:
excluded = [c for c in DROP_COLS if c in df.columns] + [TARGET]
if "source" in df.columns:
    excluded.append("source")

FEATURES = [c for c in df.columns if c not in excluded]

non_numeric = [c for c in FEATURES if not pd.api.types.is_numeric_dtype(df[c])]
assert not non_numeric, f"non-numeric feature columns: {non_numeric}"
for banned in DROP_COLS:                       # guard against them sneaking back in
    assert banned not in FEATURES, f"'{banned}' must not be a feature"

# ---- rows with a missing label are unusable ----
n_bad = int(df[TARGET].isna().sum())
if n_bad:
    print(f"dropping {n_bad:,} rows with a missing {TARGET}")
    df = df[df[TARGET].notna()].reset_index(drop=True)

# ---- encode the label to 0..K-1 ----
raw_y = df[TARGET]
if pd.api.types.is_float_dtype(raw_y) and np.allclose(raw_y, raw_y.round()):
    raw_y = raw_y.round().astype(np.int64)     # 1.0 / 0.0 -> 1 / 0
CLASSES     = sorted(pd.unique(raw_y).tolist())
NUM_CLASSES = len(CLASSES)
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}
IDX_TO_CLASS = {i: c for c, i in CLASS_TO_IDX.items()}
LABEL_NAMES  = [str(CLASS_NAMES.get(c, f"class {c}")) for c in CLASSES]

assert NUM_CLASSES >= 2, f"{TARGET} has a single value ({CLASSES}) - nothing to learn"
IS_BINARY = NUM_CLASSES == 2

X = df[FEATURES].astype(np.float32)
y = raw_y.map(CLASS_TO_IDX).astype(np.int16)

print(f"features ({len(FEATURES)}): {FEATURES}")
print(f"excluded            : {excluded}")
print(f"X: {X.shape}    y: {y.shape}")
print(f"\ntask       : {'BINARY' if IS_BINARY else f'MULTI-CLASS ({NUM_CLASSES} classes)'}")
print(f"raw values : {CLASSES}")
print(f"encoded as : {[CLASS_TO_IDX[c] for c in CLASSES]}  ->  {LABEL_NAMES}")

counts = y.value_counts().sort_index()
for i, n in counts.items():
    print(f"  {LABEL_NAMES[i]:<20} idx {i}  {n:>10,}  ({n / len(y):6.2%})")
imbalance = counts.max() / max(counts.min(), 1)
print(f"imbalance ratio (max/min): {imbalance:.1f}x")

n_nan = int(X.isna().sum().sum())
print(f"NaNs in X: {n_nan}" + ("  (XGBoost handles these natively)" if n_nan else ""))

features (22): ['B01', 'B02', 'B03', 'B04', 'B05', 'B06', 'B07', 'B08', 'B8A', 'B09', 'B11', 'B12', 'EVI', 'NDVI', 'SAVI', 'NDBI', 'NDMI', 'MNDWI', 'NDWI', 'BSI', 'BI', 'AWEI']
excluded            : ['Longitude', 'Latitude', 'ClassID', 'soilClassifiedId']
X: (3633185, 22)    y: (3633185,)

task       : BINARY
raw values : [0, 1]
encoded as : [0, 1]  ->  ['class 0', 'class 1']
  class 0              idx 0   1,439,443  (39.62%)
  class 1              idx 1   2,193,742  (60.38%)
imbalance ratio (max/min): 1.5x
NaNs in X: 0


## 4. Train / validation / test split

In [5]:
def stratified_split(X, y):
    idx = np.arange(len(y))
    strat = y if y.value_counts().min() >= 2 else None
    if strat is None:
        print("WARNING - a class has <2 rows; splitting without stratification")
    tr_idx, te_idx = train_test_split(idx, test_size=TEST_SIZE, stratify=strat,
                                      random_state=SEED)
    rel_val = VAL_SIZE / (1.0 - TEST_SIZE)
    sub = y.iloc[tr_idx]
    strat2 = sub if sub.value_counts().min() >= 2 else None
    tr_idx, va_idx = train_test_split(tr_idx, test_size=rel_val, stratify=strat2,
                                      random_state=SEED)
    return tr_idx, va_idx, te_idx


def spatial_block_split(X, y, frame):
    """Assign whole grid cells to a single split so neighbouring pixels can't straddle it."""
    need = {"Longitude", "Latitude"}
    if not need <= set(frame.columns):
        raise ValueError("spatial_block needs Longitude/Latitude in the dataframe")
    cell = (frame["Longitude"] / BLOCK_SIZE_DEG).round().astype(int).astype(str) + "_" + \
           (frame["Latitude"] / BLOCK_SIZE_DEG).round().astype(int).astype(str)
    groups = cell.values
    idx = np.arange(len(y))
    gss = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=SEED)
    tr_idx, te_idx = next(gss.split(idx, y, groups))
    rel_val = VAL_SIZE / (1.0 - TEST_SIZE)
    gss2 = GroupShuffleSplit(n_splits=1, test_size=rel_val, random_state=SEED)
    sub_tr, sub_va = next(gss2.split(tr_idx, y.iloc[tr_idx], groups[tr_idx]))
    print(f"spatial blocks: {len(np.unique(groups)):,} cells of {BLOCK_SIZE_DEG} deg")
    return tr_idx[sub_tr], tr_idx[sub_va], te_idx


if SPLIT_STRATEGY == "spatial_block":
    tr_idx, va_idx, te_idx = spatial_block_split(X, y, df)
elif SPLIT_STRATEGY == "stratified":
    tr_idx, va_idx, te_idx = stratified_split(X, y)
    print("NOTE - random split. Neighbouring pixels are near-duplicates, so scores here are")
    print("       optimistic. Set SPLIT_STRATEGY='spatial_block' for an honest estimate.")
else:
    raise ValueError(f"unknown SPLIT_STRATEGY: {SPLIT_STRATEGY}")

X_tr, y_tr = X.iloc[tr_idx], y.iloc[tr_idx]
X_va, y_va = X.iloc[va_idx], y.iloc[va_idx]
X_te, y_te = X.iloc[te_idx], y.iloc[te_idx]

assert len(set(tr_idx) & set(va_idx)) == 0 and len(set(tr_idx) & set(te_idx)) == 0 \
       and len(set(va_idx) & set(te_idx)) == 0, "splits overlap"

split_table = pd.DataFrame(
    {name: pd.Series(part).value_counts().reindex(range(NUM_CLASSES), fill_value=0)
     for name, part in (("train", y_tr), ("val", y_va), ("test", y_te))})
split_table.index = LABEL_NAMES
split_table.loc["TOTAL"] = split_table.sum()
display(split_table)

missing = [n for n, part in (("val", y_va), ("test", y_te))
           if part.nunique() < NUM_CLASSES]
if missing:
    print(f"WARNING - {missing} do not contain every class; some metrics will be undefined.")

NOTE - random split. Neighbouring pixels are near-duplicates, so scores here are
       optimistic. Set SPLIT_STRATEGY='spatial_block' for an honest estimate.


,train,val,test
class 0,1007610,215916,215917
class 1,1535619,329062,329061
TOTAL,2543229,544978,544978


## 5. XGBoost setup (task + GPU auto-detect)

In [6]:
def gpu_available():
    if not USE_GPU:
        return False
    try:
        import subprocess
        r = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True, timeout=20)
        return r.returncode == 0 and "GPU" in r.stdout
    except Exception:
        return False


on_gpu = gpu_available()
params = dict(PARAMS)

# device/tree_method spelling changed in xgboost 2.0
if XGB_VERSION >= (2, 0):
    params["tree_method"] = "hist"
    params["device"] = "cuda" if on_gpu else "cpu"
else:
    params["tree_method"] = "gpu_hist" if on_gpu else "hist"

# ---- objective, metrics and class balancing follow the detected task ----
w_tr = None
if IS_BINARY:
    params["objective"] = "binary:logistic"
    EARLY_STOP_METRIC, EXTRA_METRICS = BINARY_EARLY_STOP_METRIC, BINARY_EXTRA_METRICS
    n_pos = int((y_tr == 1).sum())
    n_neg = int(len(y_tr) - n_pos)
    if BALANCE_CLASSES:
        params["scale_pos_weight"] = n_neg / max(n_pos, 1)
    print(f"objective         : binary:logistic")
    print(f"class balance     : {n_neg:,} x '{LABEL_NAMES[0]}' / {n_pos:,} x '{LABEL_NAMES[1]}'"
          + (f"   scale_pos_weight={params['scale_pos_weight']:.4f}" if BALANCE_CLASSES else ""))
else:
    params["objective"] = "multi:softprob"
    params["num_class"] = NUM_CLASSES
    EARLY_STOP_METRIC, EXTRA_METRICS = MULTI_EARLY_STOP_METRIC, MULTI_EXTRA_METRICS
    if BALANCE_CLASSES:
        cnt = np.bincount(y_tr.to_numpy(), minlength=NUM_CLASSES)
        cw = len(y_tr) / (NUM_CLASSES * np.maximum(cnt, 1))
        w_tr = cw[y_tr.to_numpy()]
        print(f"objective         : multi:softprob (num_class={NUM_CLASSES})")
        print(f"class weights     : {dict(zip(LABEL_NAMES, np.round(cw, 4)))}")
    else:
        print(f"objective         : multi:softprob (num_class={NUM_CLASSES}), unweighted")

# the metric used for early stopping must be LAST in the list
metrics = [m for m in EXTRA_METRICS if m != EARLY_STOP_METRIC] + [EARLY_STOP_METRIC]
params["eval_metric"] = metrics
MAXIMIZE = EARLY_STOP_METRIC not in ("logloss", "mlogloss", "error", "merror", "rmse", "mae")

dtrain = xgb.DMatrix(X_tr, label=y_tr, weight=w_tr, feature_names=FEATURES)
dval   = xgb.DMatrix(X_va, label=y_va, feature_names=FEATURES)
dtest  = xgb.DMatrix(X_te, label=y_te, feature_names=FEATURES)

print(f"device            : {'GPU (cuda)' if on_gpu else 'CPU'}")
print(f"eval metrics      : {metrics}   early stop on '{EARLY_STOP_METRIC}' (maximize={MAXIMIZE})")
print(f"params            : {json.dumps(params, default=str)}")

objective         : binary:logistic
class balance     : 1,007,610 x 'class 0' / 1,535,619 x 'class 1'   scale_pos_weight=0.6562
device            : GPU (cuda)
eval metrics      : ['logloss', 'auc', 'aucpr']   early stop on 'aucpr' (maximize=True)
params            : {"max_depth": 6, "eta": 0.05, "subsample": 0.8, "colsample_bytree": 0.8, "min_child_weight": 1, "gamma": 0.0, "lambda": 1.0, "alpha": 0.0, "seed": 42, "tree_method": "hist", "device": "cuda", "objective": "binary:logistic", "scale_pos_weight": 0.656158851902718, "eval_metric": ["logloss", "auc", "aucpr"]}


## 6. Best-weight checkpoint callback

XGBoost's `save_model` writes **every** tree built so far. If you save after training
finishes, the file contains the extra `EARLY_STOP_ROUNDS` trees that made validation *worse* -
and a reloaded booster does not skip them unless you remember to pass `iteration_range`.

Two things guard against that here:

1. The callback below saves the booster **at the moment the validation metric improves**, so
   the file never contains trees past the best iteration.
2. After training the booster is sliced with `booster[:best_iteration + 1]` and saved again.

Both files are then reloaded and checked to produce identical predictions.

In [7]:
class BestModelCheckpoint(xgb.callback.TrainingCallback):
    """Persist the booster every time the watched validation metric improves."""

    def __init__(self, path, eval_set, metric, maximize=True, verbose=True):
        self.path, self.eval_set, self.metric = str(path), eval_set, metric
        self.maximize, self.verbose = maximize, verbose
        self.best_score = -np.inf if maximize else np.inf
        self.best_iteration = -1
        self.n_saves = 0

    def _improved(self, score):
        return score > self.best_score if self.maximize else score < self.best_score

    def after_iteration(self, model, epoch, evals_log):
        try:
            score = float(evals_log[self.eval_set][self.metric][-1])
        except (KeyError, IndexError):
            return False
        if self._improved(score):
            self.best_score, self.best_iteration = score, epoch
            model.save_model(self.path)          # <- best weights hit disk here
            self.n_saves += 1
            if self.verbose and self.n_saves % 25 == 0:
                print(f"    [ckpt] iter {epoch:>5}  {self.metric}={score:.6f}"
                      f"  -> {Path(self.path).name}")
        return False   # never stop training; early stopping owns that


best_ckpt_path = Path(CKPT_DIR) / BEST_MODEL
print(f"checkpoint target: {best_ckpt_path}")

checkpoint target: /kaggle/working/checkpoints/best_model.json


## 7. Train

In [8]:
def _fit(p):
    """One training run. Returns (booster, evals_result, callback)."""
    cb = BestModelCheckpoint(best_ckpt_path, "val", EARLY_STOP_METRIC, maximize=MAXIMIZE)
    ev = {}
    b = xgb.train(
        p,
        dtrain,
        num_boost_round=NUM_BOOST_ROUND,
        evals=[(dtrain, "train"), (dval, "val")],   # early stopping watches the LAST entry
        early_stopping_rounds=EARLY_STOP_ROUNDS,
        evals_result=ev,
        callbacks=[cb],
        verbose_eval=50,
    )
    return b, ev, cb


t0 = time.time()
try:
    booster, evals_result, checkpoint_cb = _fit(params)
except Exception as exc:
    if not on_gpu:
        raise
    # a GPU is visible but unusable (driver / build mismatch) - fall back rather than die
    print(f"\nGPU training failed ({type(exc).__name__}: {exc})")
    print("retrying on CPU ...\n")
    on_gpu = False
    if XGB_VERSION >= (2, 0):
        params["device"] = "cpu"
    else:
        params["tree_method"] = "hist"
    t0 = time.time()
    booster, evals_result, checkpoint_cb = _fit(params)
elapsed = time.time() - t0

best_iter = int(getattr(booster, "best_iteration", checkpoint_cb.best_iteration))

print(f"\ntrained in {elapsed:.1f}s on {'GPU' if on_gpu else 'CPU'}")
print(f"boosting rounds      : {booster.num_boosted_rounds()}")
print(f"best_iteration       : {best_iter}")
print(f"best val {EARLY_STOP_METRIC:<12}: {checkpoint_cb.best_score:.6f}")
print(f"checkpoint saves     : {checkpoint_cb.n_saves}")

assert best_iter == checkpoint_cb.best_iteration, (
    f"callback best ({checkpoint_cb.best_iteration}) != booster best ({best_iter})")
assert best_ckpt_path.exists(), "no checkpoint was written"

[0]	train-logloss:0.65705	train-auc:0.96160	train-aucpr:0.97267	val-logloss:0.65709	val-auc:0.96156	val-aucpr:0.97264
    [ckpt] iter    24  aucpr=0.989168  -> best_model.json
    [ckpt] iter    49  aucpr=0.992765  -> best_model.json
[50]	train-logloss:0.16443	train-auc:0.98998	train-aucpr:0.99313	val-logloss:0.16504	val-auc:0.98977	val-aucpr:0.99292
    [ckpt] iter    74  aucpr=0.994617  -> best_model.json
    [ckpt] iter    99  aucpr=0.995545  -> best_model.json
[100]	train-logloss:0.10887	train-auc:0.99361	train-aucpr:0.99581	val-logloss:0.11009	val-auc:0.99333	val-aucpr:0.99557
    [ckpt] iter   124  aucpr=0.996161  -> best_model.json
    [ckpt] iter   149  aucpr=0.996589  -> best_model.json
[150]	train-logloss:0.09258	train-auc:0.99507	train-aucpr:0.99680	val-logloss:0.09419	val-auc:0.99479	val-aucpr:0.99660
    [ckpt] iter   174  aucpr=0.996874  -> best_model.json
    [ckpt] iter   199  aucpr=0.997123  -> best_model.json
[200]	train-logloss:0.08366	train-auc:0.99586	train-aucpr:0

In [9]:
# slice off the rounds built after the best iteration, then save
final_model_path = Path(CKPT_DIR) / FINAL_MODEL
best_booster = booster[: best_iter + 1]
best_booster.save_model(final_model_path)

print(f"full booster : {booster.num_boosted_rounds()} rounds")
print(f"sliced       : {best_booster.num_boosted_rounds()} rounds -> {final_model_path}")
print(f"best ckpt    : {best_ckpt_path} ({best_ckpt_path.stat().st_size / 1024:.1f} KB)")

full booster : 3000 rounds
sliced       : 3000 rounds -> /kaggle/working/checkpoints/final_model.json
best ckpt    : /kaggle/working/checkpoints/best_model.json (20141.1 KB)


In [10]:
# training curves
fig, axes = plt.subplots(1, len(metrics), figsize=(5.5 * len(metrics), 4), squeeze=False)
for ax, m in zip(axes[0], metrics):
    ax.plot(evals_result["train"][m], label="train", lw=1.4)
    ax.plot(evals_result["val"][m], label="val", lw=1.4)
    ax.axvline(best_iter, color="crimson", ls="--", lw=1, label=f"best iter {best_iter}")
    ax.set_xlabel("boosting round"); ax.set_ylabel(m); ax.set_title(m)
    ax.legend(); ax.grid(alpha=.3)
plt.tight_layout()
plt.savefig(Path(CKPT_DIR) / "training_curves.png", dpi=130, bbox_inches="tight")
plt.show()

## 8. Verify the checkpoint reloads correctly

In [11]:
# A checkpoint you never reload is a checkpoint you can't trust.
reloaded = xgb.Booster()
reloaded.load_model(best_ckpt_path)

p_sliced   = best_booster.predict(dtest)
p_reloaded = reloaded.predict(dtest)
# the full booster needs an explicit range to ignore the post-best rounds
p_full_ok  = booster.predict(dtest, iteration_range=(0, best_iter + 1))
p_full_bad = booster.predict(dtest)   # what you'd get by forgetting iteration_range

print(f"reloaded rounds                      : {reloaded.num_boosted_rounds()}")
print(f"max |sliced - reloaded_checkpoint|   : {np.abs(p_sliced - p_reloaded).max():.3e}")
print(f"max |sliced - full[:best+1]|         : {np.abs(p_sliced - p_full_ok).max():.3e}")
print(f"max |sliced - full(no range)|        : {np.abs(p_sliced - p_full_bad).max():.3e}"
      f"  <- non-zero is the bug this guards against")

np.testing.assert_allclose(p_sliced, p_reloaded, rtol=1e-5, atol=1e-6)
np.testing.assert_allclose(p_sliced, p_full_ok, rtol=1e-5, atol=1e-6)
print("\nOK - the saved checkpoint reproduces the best model exactly")

model = reloaded   # everything below uses the reloaded checkpoint

reloaded rounds                      : 3000
max |sliced - reloaded_checkpoint|   : 1.192e-07
max |sliced - full[:best+1]|         : 0.000e+00
max |sliced - full(no range)|        : 0.000e+00  <- non-zero is the bug this guards against

OK - the saved checkpoint reproduces the best model exactly


## 9. Decision rule

* **Binary** - the probability cut-off is tuned to maximise F1 **on validation only**;
  the test set stays untouched until the next section.
* **Multi-class** - the predicted class is `argmax` over `multi:softprob` outputs, so
  there is no threshold to tune.

In [12]:
def as_matrix(proba):
    """Return an (n, K) probability matrix for either task."""
    return np.column_stack([1.0 - proba, proba]) if IS_BINARY else proba


def to_pred(proba):
    """Class indices (0..K-1) from raw model output."""
    if IS_BINARY:
        return (proba >= THRESHOLD).astype(int)
    return proba.argmax(axis=1)


p_val = model.predict(dval)

if IS_BINARY and TUNE_THRESHOLD:
    grid = np.linspace(0.05, 0.95, 181)
    f1s = [f1_score(y_va, (p_val >= t).astype(int), zero_division=0) for t in grid]
    THRESHOLD = float(grid[int(np.argmax(f1s))])
    print(f"tuned threshold : {THRESHOLD:.3f}   "
          f"(val F1 = {max(f1s):.4f}, vs {f1s[len(grid) // 2]:.4f} at 0.50)")
elif IS_BINARY:
    THRESHOLD = 0.5
    print("threshold fixed at 0.50")
else:
    THRESHOLD = None
    print(f"multi-class ({NUM_CLASSES} classes) - predicting argmax, no threshold to tune")

tuned threshold : 0.405   (val F1 = 0.9876, vs 0.9874 at 0.50)


## 10. Test-set evaluation

In [13]:
AVG = "binary" if IS_BINARY else "macro"


def _auc(y_true, proba):
    try:
        if IS_BINARY:
            return float(roc_auc_score(y_true, proba))
        return float(roc_auc_score(y_true, proba, multi_class="ovr", average="macro",
                                   labels=list(range(NUM_CLASSES))))
    except ValueError:
        return float("nan")          # a class missing from this split


def _ap(y_true, proba):
    try:
        if IS_BINARY:
            return float(average_precision_score(y_true, proba))
        oh = np.eye(NUM_CLASSES)[np.asarray(y_true)]
        return float(average_precision_score(oh, as_matrix(proba), average="macro"))
    except ValueError:
        return float("nan")


def evaluate(name, y_true, proba):
    pred = to_pred(proba)
    return {
        "split":     name,
        "n":         int(len(y_true)),
        "accuracy":  round(float(accuracy_score(y_true, pred)), 4),
        "precision": round(float(precision_score(y_true, pred, average=AVG, zero_division=0)), 4),
        "recall":    round(float(recall_score(y_true, pred, average=AVG, zero_division=0)), 4),
        "f1":        round(float(f1_score(y_true, pred, average=AVG, zero_division=0)), 4),
        "mcc":       round(float(matthews_corrcoef(y_true, pred)), 4),
        "roc_auc":   round(_auc(y_true, proba), 4),
        "pr_auc":    round(_ap(y_true, proba), 4),
    }


p_train = model.predict(dtrain)
p_test  = model.predict(dtest)

results = pd.DataFrame([
    evaluate("train", y_tr, p_train),
    evaluate("val",   y_va, p_val),
    evaluate("test",  y_te, p_test),
])
print(f"averaging: {AVG}" + (f"   threshold: {THRESHOLD:.3f}" if IS_BINARY else "   rule: argmax"))
display(results)

print("\nTEST classification report")
print(classification_report(y_te, to_pred(p_test), labels=list(range(NUM_CLASSES)),
                            target_names=LABEL_NAMES, digits=4, zero_division=0))

gap = results.loc[0, "f1"] - results.loc[2, "f1"]
if gap > 0.05:
    print(f"NOTE - train F1 exceeds test F1 by {gap:.3f}; consider more regularisation "
          f"(max_depth, lambda, min_child_weight) or a spatial split.")

averaging: binary   threshold: 0.405


,split,n,accuracy,precision,recall,f1,mcc,roc_auc,pr_auc
0,train,2543229,0.9888,0.9920,0.9895,0.9907,0.9766,0.9993,0.9996
1,val,544978,0.9851,0.9886,0.9867,0.9876,0.9688,0.9988,0.9992
2,test,544978,0.9850,0.9885,0.9866,0.9875,0.9686,0.9988,0.9992



TEST classification report
              precision    recall  f1-score   support

     class 0     0.9796    0.9825    0.9811    215917
     class 1     0.9885    0.9866    0.9875    329061

    accuracy                         0.9850    544978
   macro avg     0.9841    0.9846    0.9843    544978
weighted avg     0.9850    0.9850    0.9850    544978



In [14]:
cm = confusion_matrix(y_te, to_pred(p_test), labels=list(range(NUM_CLASSES)))
proba_te = as_matrix(p_test)
curve_classes = [1] if IS_BINARY else list(range(NUM_CLASSES))

fig, ax = plt.subplots(1, 3, figsize=(17, 4.8))

ax[0].imshow(cm, cmap="Blues")
ax[0].set_xticks(range(NUM_CLASSES)); ax[0].set_yticks(range(NUM_CLASSES))
ax[0].set_xticklabels(LABEL_NAMES, rotation=30, ha="right")
ax[0].set_yticklabels(LABEL_NAMES)
ax[0].set_xlabel("predicted"); ax[0].set_ylabel("true")
for (i, j), v in np.ndenumerate(cm):
    ax[0].text(j, i, f"{v:,}", ha="center", va="center",
               color="white" if v > cm.max() / 2 else "black",
               fontsize=12 if NUM_CLASSES <= 4 else 9)
ax[0].set_title("Confusion matrix (test)"
                + (f", thr={THRESHOLD:.2f}" if IS_BINARY else ""))

for k in curve_classes:
    yk = (np.asarray(y_te) == k).astype(int)
    if yk.sum() == 0:
        continue
    fpr, tpr, _ = roc_curve(yk, proba_te[:, k])
    prec, rec, _ = precision_recall_curve(yk, proba_te[:, k])
    lab = LABEL_NAMES[k]
    ax[1].plot(fpr, tpr, lw=1.7, label=f"{lab} (AUC={roc_auc_score(yk, proba_te[:, k]):.4f})")
    ax[2].plot(rec, prec, lw=1.7,
               label=f"{lab} (AP={average_precision_score(yk, proba_te[:, k]):.4f})")

ax[1].plot([0, 1], [0, 1], "k--", lw=.8)
ax[1].set_xlabel("FPR"); ax[1].set_ylabel("TPR")
ax[1].set_title("ROC (test)" + ("" if IS_BINARY else ", one-vs-rest"))
ax[1].legend(fontsize=8); ax[1].grid(alpha=.3)

ax[2].set_xlabel("recall"); ax[2].set_ylabel("precision")
ax[2].set_title("Precision-Recall (test)" + ("" if IS_BINARY else ", one-vs-rest"))
ax[2].legend(fontsize=8); ax[2].grid(alpha=.3)

plt.tight_layout()
plt.savefig(Path(CKPT_DIR) / "evaluation.png", dpi=130, bbox_inches="tight")
plt.show()

per_class = pd.DataFrame({
    "class":   LABEL_NAMES,
    "raw_id":  [IDX_TO_CLASS[i] for i in range(NUM_CLASSES)],
    "support": cm.sum(axis=1),
    "correct": np.diag(cm),
})
per_class["recall"] = (per_class["correct"] / per_class["support"].replace(0, np.nan)).round(4)
display(per_class)

,class,raw_id,support,correct,recall
0,class 0,0,215917,212139,0.9825
1,class 1,1,329061,324653,0.9866


## 11. Feature importance

In [15]:
gain   = model.get_score(importance_type="gain")
weight = model.get_score(importance_type="weight")
cover  = model.get_score(importance_type="cover")

imp = pd.DataFrame({
    "feature": FEATURES,
    "gain":    [gain.get(f, 0.0)   for f in FEATURES],
    "weight":  [weight.get(f, 0.0) for f in FEATURES],
    "cover":   [cover.get(f, 0.0)  for f in FEATURES],
}).sort_values("gain", ascending=False).reset_index(drop=True)
imp["gain_pct"] = (imp["gain"] / max(imp["gain"].sum(), 1e-12) * 100).round(2)
display(imp)

top = imp.head(20).iloc[::-1]
plt.figure(figsize=(8, max(4, 0.34 * len(top))))
plt.barh(top["feature"], top["gain"], color="#2b6cb0")
plt.xlabel("gain"); plt.title("Feature importance (gain)"); plt.grid(axis="x", alpha=.3)
plt.tight_layout()
plt.savefig(Path(CKPT_DIR) / "feature_importance.png", dpi=130, bbox_inches="tight")
plt.show()

for banned in DROP_COLS:
    assert banned not in imp["feature"].values, f"'{banned}' leaked into the model"
print(f"confirmed: {DROP_COLS} are absent from the model")

,feature,gain,weight,cover,gain_pct
0,AWEI,768.705811,4987.0,4081.042725,25.10
1,MNDWI,340.291809,4935.0,2520.972900,11.11
2,BSI,314.297455,5587.0,6176.933594,10.26
3,B01,306.541138,15945.0,3851.144775,10.01
4,B03,242.945465,5625.0,6284.093750,7.93
5,B02,231.725525,7651.0,5303.014648,7.56
6,BI,132.292023,3074.0,2551.704346,4.32
7,B05,101.588440,12894.0,3730.610596,3.32
8,B11,94.581398,11759.0,3466.901855,3.09
9,NDWI,74.949486,5706.0,4560.979492,2.45


confirmed: ['Longitude', 'Latitude', 'ClassID'] are absent from the model


## 12. Save artifacts

In [16]:
artifacts = {
    "trained_at":        time.strftime("%Y-%m-%d %H:%M:%S"),
    "xgboost_version":   xgb.__version__,
    "device":            "cuda" if on_gpu else "cpu",
    "target":            TARGET,
    "task":              "binary" if IS_BINARY else "multiclass",
    "num_classes":       int(NUM_CLASSES),
    "classes_raw":       [int(c) if isinstance(c, (int, np.integer)) else str(c)
                          for c in CLASSES],
    "class_names":       LABEL_NAMES,
    "idx_to_class":      {str(i): (int(c) if isinstance(c, (int, np.integer)) else str(c))
                          for i, c in IDX_TO_CLASS.items()},
    "features":          FEATURES,
    "n_features":        len(FEATURES),
    "excluded_columns":  excluded,
    "split_strategy":    SPLIT_STRATEGY,
    "split_sizes":       {"train": int(len(y_tr)), "val": int(len(y_va)), "test": int(len(y_te))},
    "params":            {k: (v if isinstance(v, (int, float, str, list)) else str(v))
                          for k, v in params.items()},
    "num_boost_round":   NUM_BOOST_ROUND,
    "early_stop_rounds": EARLY_STOP_ROUNDS,
    "early_stop_metric": EARLY_STOP_METRIC,
    "best_iteration":    int(best_iter),
    f"best_val_{EARLY_STOP_METRIC}": round(float(checkpoint_cb.best_score), 6),
    "threshold":         (round(float(THRESHOLD), 4) if THRESHOLD is not None else None),
    "metrics":           results.to_dict(orient="records"),
    "top_features":      imp.head(10)[["feature", "gain_pct"]].to_dict(orient="records"),
}

(Path(CKPT_DIR) / "run_metadata.json").write_text(json.dumps(artifacts, indent=2))
imp.to_csv(Path(CKPT_DIR) / "feature_importance.csv", index=False)
results.to_csv(Path(CKPT_DIR) / "metrics.csv", index=False)
per_class.to_csv(Path(CKPT_DIR) / "per_class_test.csv", index=False)

print(f"artifacts in {CKPT_DIR}:")
for p in sorted(Path(CKPT_DIR).iterdir()):
    print(f"  {p.name:<28}{p.stat().st_size / 1024:>10.1f} KB")

artifacts in /kaggle/working/checkpoints:
  best_model.json                20141.1 KB
  evaluation.png                    78.4 KB
  feature_importance.csv             1.1 KB
  feature_importance.png            36.5 KB
  final_model.json               20141.0 KB
  metrics.csv                        0.2 KB
  per_class_test.csv                 0.1 KB
  run_metadata.json                  2.6 KB
  training_curves.png               83.4 KB


## 13. How to load the checkpoint later

In [17]:
# Copy-paste this to use the trained model in another notebook.
_SNIPPET_LINES = [
    "import json, numpy as np, pandas as pd, xgboost as xgb",
    "",
    "meta  = json.load(open(\"__CKPT__/run_metadata.json\"))",
    "model = xgb.Booster()",
    "model.load_model(\"__CKPT__/__BEST__\")",
    "",
    "FEATURES    = meta[\"features\"]        # no Longitude / Latitude / ClassID",
    "IDX_TO_CLASS = {int(k): v for k, v in meta[\"idx_to_class\"].items()}",
    "IS_BINARY   = meta[\"task\"] == \"binary\"",
    "THRESHOLD   = meta[\"threshold\"]       # None for multi-class",
    "",
    "def predict(frame):",
    "    X = frame[FEATURES].astype(np.float32)",
    "    d = xgb.DMatrix(X, feature_names=FEATURES)",
    "    proba = model.predict(d)   # no iteration_range needed - the checkpoint",
    "                               # holds ONLY the best rounds",
    "    if IS_BINARY:",
    "        idx = (proba >= THRESHOLD).astype(int)",
    "        conf = np.where(idx == 1, proba, 1.0 - proba)",
    "    else:",
    "        idx = proba.argmax(axis=1)",
    "        conf = proba.max(axis=1)",
    "    return pd.DataFrame({\"__TARGET__\": [IDX_TO_CLASS[i] for i in idx],",
    "                         \"confidence\": conf})",
]
LOAD_SNIPPET = chr(10).join(_SNIPPET_LINES) \
    .replace('__CKPT__', str(CKPT_DIR)) \
    .replace('__BEST__', BEST_MODEL) \
    .replace('__TARGET__', TARGET)
print(LOAD_SNIPPET)
Path(CKPT_DIR, 'load_model_snippet.py').write_text(LOAD_SNIPPET + chr(10))

# prove the checkpoint round-trips, here and now
_m = xgb.Booster()
_m.load_model(Path(CKPT_DIR) / BEST_MODEL)
_d = xgb.DMatrix(X_te[FEATURES].astype(np.float32), feature_names=FEATURES)
_p = _m.predict(_d)
print(f'\nround-trip check on test set -> max diff {np.abs(_p - p_test).max():.3e}')
assert np.allclose(_p, p_test, rtol=1e-5, atol=1e-6)
print('load snippet verified')

import json, numpy as np, pandas as pd, xgboost as xgb

meta  = json.load(open("/kaggle/working/checkpoints/run_metadata.json"))
model = xgb.Booster()
model.load_model("/kaggle/working/checkpoints/best_model.json")

FEATURES    = meta["features"]        # no Longitude / Latitude / ClassID
IDX_TO_CLASS = {int(k): v for k, v in meta["idx_to_class"].items()}
IS_BINARY   = meta["task"] == "binary"
THRESHOLD   = meta["threshold"]       # None for multi-class

def predict(frame):
    X = frame[FEATURES].astype(np.float32)
    d = xgb.DMatrix(X, feature_names=FEATURES)
    proba = model.predict(d)   # no iteration_range needed - the checkpoint
                               # holds ONLY the best rounds
    if IS_BINARY:
        idx = (proba >= THRESHOLD).astype(int)
        conf = np.where(idx == 1, proba, 1.0 - proba)
    else:
        idx = proba.argmax(axis=1)
        conf = proba.max(axis=1)
    return pd.DataFrame({"soilClassifiedId": [IDX_TO_CLASS[i] for i in idx],
                     

---
### Notes

* **`soilClassifiedId`** is read as-is. Section 3 prints the distinct values it found and how
  they were encoded - check that line matches what you intended before trusting any score.
  If the column has more than two values the notebook switches to `multi:softprob`
  automatically; nothing needs editing.
* **`CLASS_NAMES`** in section 1 is cosmetic - fill it in (e.g. `{0: "urban", 1: "bare soil"}`)
  and the reports, confusion matrix and ROC legends use those names instead of `class 0`/`class 1`.
* **`SPLIT_STRATEGY`** defaults to `"stratified"`. Satellite pixels next to each other are
  near-duplicates, so a random split puts near-copies of training rows into the test set and
  inflates every score. Re-run with `"spatial_block"` before you quote a number as the model's
  real-world accuracy - expect it to drop, and that drop is the honest part.
* **`best_model.json` vs `final_model.json`** - identical content by construction. The first is
  written mid-training on every improvement (so a killed session still leaves the best weights
  on disk); the second is the sliced booster written at the end. Section 8 asserts they agree.
* **Threshold** is tuned on validation, never on test, and only for the binary case.
* **Class imbalance** is handled by `scale_pos_weight` (binary) or per-row sample weights
  (multi-class), both computed from the training split only. Set `BALANCE_CLASSES = False`
  to train unweighted.
* To download the trained model from Kaggle: `/kaggle/working/` is saved as the run's output -
  grab `checkpoints/best_model.json` from the Output tab.